## Using the Exsisting code used in files

In [1]:
from qdrant_client import QdrantClient, models
import os
from sentence_transformers import CrossEncoder, SentenceTransformer
import  numpy as np

In [2]:
DEFAULT_PATH = "D:\\M.tech\\Research Area\\RAG pipeline\\DB"
TEXT_CANDIDATES = ("text", "page_content", "content", "chunk_text", "chunk")

In [3]:
from dataclasses import dataclass, field


@dataclass
class Hit:
    id: str
    text: str
    payload: dict
    dense_rank: int | None = None
    bm25_rank: int | None = None
    rrf_score: float = 0.0
    rerank_score: float = 0.0
@dataclass
class Timings:
    ms: dict = field(default_factory=dict)

In [4]:
## Find which DB present and initialize them
def make_client(path: str = DEFAULT_PATH) -> QdrantClient:
    if not os.path.isdir(path):
        raise SystemExit(f"Qdrant folder not found: {path}\nPass the folder you used with QdrantClient(path=...) via --path.")
    try:
        client = QdrantClient(path=path)
        names = [c.name for c in client.get_collections().collections]
    except Exception as e:
        # local mode allows one process at a time: close notebooks / other scripts that have the folder open
        raise SystemExit(f"Could not open {path}: {e}\nIf another notebook or script has this folder open, close it first.")
    print(f"Opened {path}; collections: {names}")
    return client

In [5]:
found = []
for d, dirs, files in os.walk(DEFAULT_PATH):
    if "meta.json" in files:
        found.append(d)
        dirs[:] = []  # do not descend into a storage folder
print(sorted(found))

['D:\\M.tech\\Research Area\\RAG pipeline\\DB\\BGE-M3_Embedding\\ContextualTokenAwareChunking', 'D:\\M.tech\\Research Area\\RAG pipeline\\DB\\BGE-M3_Embedding\\HierarchicalChunking', 'D:\\M.tech\\Research Area\\RAG pipeline\\DB\\BGE-M3_Embedding\\StructureAwareChunking', 'D:\\M.tech\\Research Area\\RAG pipeline\\DB\\bge-small-en-v1.5_Embedding\\ContextualTokenAwareChunking', 'D:\\M.tech\\Research Area\\RAG pipeline\\DB\\bge-small-en-v1.5_Embedding\\HierarchicalChunking', 'D:\\M.tech\\Research Area\\RAG pipeline\\DB\\bge-small-en-v1.5_Embedding\\StructureAwareChunking', 'D:\\M.tech\\Research Area\\RAG pipeline\\DB\\multilingual-E5-base_Embedding\\ContextualTokenAwareChunking', 'D:\\M.tech\\Research Area\\RAG pipeline\\DB\\multilingual-E5-base_Embedding\\HierarchicalChunking', 'D:\\M.tech\\Research Area\\RAG pipeline\\DB\\multilingual-E5-base_Embedding\\StructureAwareChunking']


In [6]:
for db in sorted(found)[:1]:
    print(db)


D:\M.tech\Research Area\RAG pipeline\DB\BGE-M3_Embedding\ContextualTokenAwareChunking


In [7]:
def flat(payload: dict | None) -> dict:
    """LangChain-style payloads nest section/source under 'metadata'; lift them to the top level."""
    try:
        payload = payload or {}
        meta = payload.get("metadata")
    except Exception as e:
        pass
    return {**meta, **payload} if isinstance(meta, dict) else payload

In [8]:
def autodetect_fields(client: QdrantClient, collection: str):
    """Set the vector name and text field for THIS collection (your 9 DBs may differ). Returns (vector_size, payload_keys, n_points)."""
    try:
        global TEXT_FIELD, VECTOR_NAME
        info = client.get_collection(collection)
        vecs = info.config.params.vectors
        if isinstance(vecs, dict):  # named vectors
            VECTOR_NAME = next(iter(vecs))
            size = vecs[VECTOR_NAME].size
        else:
            VECTOR_NAME, size = None, vecs.size
        pts, _ = client.scroll(collection, limit=1, with_payload=True, with_vectors=False)
        keys = list(flat(pts[0].payload).keys()) if pts else []
        for c in (TEXT_FIELD, TEXT_CANDIDATES):
            if c in keys:
                TEXT_FIELD = c
                break
        else:
            raise SystemExit(f"[{collection}] no text field found in payload keys {keys}. Set TEXT_FIELD at the top of hybrid_rag.py.")
        return size, keys, info.points_count
    except Exception as e:
        raise SystemExit(f"[{collection}] failed to autodetect field : {e}")

In [9]:
quardrant_db = make_client('D:\\M.tech\\Research Area\\RAG pipeline\\DB\\bge-small-en-v1.5_Embedding\\StructureAwareChunking')

Opened D:\M.tech\Research Area\RAG pipeline\DB\bge-small-en-v1.5_Embedding\StructureAwareChunking; collections: ['bge_small_en_v1.5_StructureAwareChunking']


In [10]:
#quardrant_db.close()

In [11]:
c = quardrant_db.get_collections().collections

In [12]:
c[0].name

'bge_small_en_v1.5_StructureAwareChunking'

In [13]:
info = quardrant_db.get_collection('bge_small_en_v1.5_StructureAwareChunking')
vecs = info.config.params.vectors

In [14]:
info

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=15863, segments_count=1, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=None, payload=None, id_tracker=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_inte

In [15]:
vecs

VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None)

In [16]:
if isinstance(vecs, dict):  # named vectors
            VECTOR_NAME = next(iter(vecs))
            size = vecs[VECTOR_NAME].size
else:
    VECTOR_NAME, size = None, vecs.size

In [17]:
VECTOR_NAME,size

(None, 384)

In [18]:
TEXT_FIELD="page_content"
VECTOR_NAME =""

In [19]:
pts, _ = quardrant_db.scroll('bge_small_en_v1.5_StructureAwareChunking', limit=1, with_payload=True, with_vectors=False)
keys = list(flat(pts[0].payload).keys()) if pts else []
'''for c in (TEXT_FIELD, TEXT_CANDIDATES):
    if c in keys:
        TEXT_FIELD = c
        break
else:
    raise SystemExit(f"[{'bge_small_en_v1.5_StructureAwareChunking'}] no text field found in payload keys {keys}. Set TEXT_FIELD at the top of hybrid_rag.py.")'''

'for c in (TEXT_FIELD, TEXT_CANDIDATES):\n    if c in keys:\n        TEXT_FIELD = c\n        break\nelse:\n    raise SystemExit(f"[{\'bge_small_en_v1.5_StructureAwareChunking\'}] no text field found in payload keys {keys}. Set TEXT_FIELD at the top of hybrid_rag.py.")'

In [20]:
keys

['heading', 'page', 'content_type', 'source', 'page_content', 'metadata']

In [21]:
ids, texts, payloads = [], [], []
offset = None
while True:
    points, offset = quardrant_db.scroll(
        collection_name='bge_small_en_v1.5_StructureAwareChunking',
        limit=1000,
        offset=offset,
        with_payload=True,
        with_vectors=False,
    )
    for p in points:
        ids.append(p.id)
        fp = flat(p.payload)
        texts.append(fp.get(TEXT_FIELD, ""))
        payloads.append(fp)
    if offset is None:
        break

In [22]:
set([load['heading'].split(">")[0] for load in payloads])

{'',
 'ECSBC 2024 ',
 'Energy CONSERVATION BUILDING CODE TIP SHEET',
 'Energy CONSERVATION BUILDING CODE TIP SHEET ',
 'Input Output Reference',
 'Input Output Reference '}

In [23]:
## Rerankar set
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device='cpu')


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [24]:
cfg = {"name": 'bge_small_en_v1.5_StructureAwareChunking', "label": ""'D:\\M.tech\\Research Area\\RAG pipeline\\DB\\bge-small-en-v1.5_Embedding\\StructureAwareChunking',
                       "embed_model": "BAAI/bge-small-en-v1.5", "query_prefix": ""}

query = "What connected load or contract demand makes a commercial building fall within the scope of ECBC 2017?"
section = None
source = None

In [25]:
import time
from concurrent.futures import ThreadPoolExecutor
import  re
from rank_bm25 import BM25Okapi
from Retrival.bm25_index import BM25Index

In [26]:
embeder = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [27]:
len(texts)

15863

In [28]:

def tokenize(text: str) -> list[str]:
    return [t.lower() for t in _TOKEN.findall(text)]

In [29]:
_TOKEN = re.compile(r"[A-Za-z0-9]+(?:[.\-/][A-Za-z0-9]+)*")
bm25 = BM25Index(texts,tokenize)
POOL: list = []  # top-5 hits of every DB, for pooled labelling (--export-pool)


In [30]:
# ---- individual retrievers -------------------------------------------
def _dense(query: str, flt: models.Filter | None) -> list[Hit]:
    vec = embeder.encode(query)
    res = quardrant_db.query_points(
        collection_name='bge_small_en_v1.5_StructureAwareChunking',
        query=vec,
        using=None,
        limit=50,
        query_filter=flt,
        with_payload=True,
    ).points
    return [
        Hit(id=str(p.id), text=flat(p.payload).get(TEXT_FIELD, ""), payload=flat(p.payload), dense_rank=r)
        for r, p in enumerate(res, start=1)
    ]

def _bm25(query: str, flt_fn) -> list[Hit]:
    mask = None
    if flt_fn:
        mask = np.fromiter((flt_fn(p) for p in payloads), dtype=bool, count=len(payloads))
    found = bm25.search(query, top_k=10, mask=mask)
    return [Hit(id=str(ids[i]), text=texts[i], payload=payloads[i], bm25_rank=r)
            for r, (i, _) in enumerate(found, start=1)]

# ---- fusion ------------------------------------------------------------
@staticmethod
def rrf(dense: list[Hit], sparse: list[Hit], k: int = 30) -> list[Hit]:
    merged: dict[str, Hit] = {}
    for hits, attr in ((dense, "dense_rank"), (sparse, "bm25_rank")):
        for h in hits:
            m = merged.setdefault(h.id, h)
            setattr(m, attr, getattr(h, attr))
            m.rrf_score += 1.0 / (k + getattr(h, attr))
    return sorted(merged.values(), key=lambda h: h.rrf_score, reverse=True)

# ---- rerank --------------------------------------------------------------
def rerank(query: str, hits: list[Hit], top_k: int = 5) -> list[Hit]:
    if not hits:
        return []
    scores = reranker.predict([(query, h.text) for h in hits], batch_size=16, show_progress_bar=False)
    for h, s in zip(hits, scores):
        h.rerank_score = float(s)
    return sorted(hits, key=lambda h: h.rerank_score, reverse=True)[:top_k]


In [31]:
 def py_filter(payload: dict) -> bool:
            return (not section or payload.get("section") == section) and (
                not source or payload.get("source") == source
            )

In [32]:
"""Optional metadata filters: section / source (matched on the payload fields above)."""
t = Timings()
conds = []
qflt = models.Filter(must=conds) if conds else None


t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=2) as ex:  # run both retrievers in parallel
    f_dense = ex.submit(_dense, query, qflt)
    f_bm25 = ex.submit(_bm25, query, py_filter if conds else None)
    dense, sparse = f_dense.result(), f_bm25.result()
t.ms["retrieve_parallel"] = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
fused = rrf(dense, sparse)[:30]
t.ms["rrf"] = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
top = rerank(query, fused, 10)
t.ms["rerank"] = (time.perf_counter() - t0) * 1000

t.ms["total"] = sum(t.ms.values())


In [33]:
top[0].payload

{'heading': 'ECSBC 2024 > 2. Scope',
 'page': 42,
 'content_type': 'text',
 'source': 'ECBC_2024.pdf',
 'page_content': 'The Code is applicable to buildings or building complexes that have a connected load of 100 kW or greater or a contract demand of 120 kVA or greater and are intended to be used for commercial or office buildings.\nBuildings intended for residential purposes are not covered under this Code.',
 'metadata': {'heading': 'ECSBC 2024 > 2. Scope',
  'page': 42,
  'content_type': 'text',
  'source': 'ECBC_2024.pdf'}}

In [34]:

for h in top:
    print(h.payload['page_content'],end=" ")
        #POOL.append((query, str(h.payload.get("source")), h.payload.get("page"),
                # h.payload.get("section") or h.payload.get("parent_title"), h.text[:200], cfg["label"]))


The Code is applicable to buildings or building complexes that have a connected load of 100 kW or greater or a contract demand of 120 kVA or greater and are intended to be used for commercial or office buildings.
Buildings intended for residential purposes are not covered under this Code. If any existing building after additions or alterations changes its connected load to 100 kilo- Watt (kW) or above or a contract demand of 120 kilo-Volt Ampere (kVA) or above shall comply with the provisions of chapter 4 through 11. Compliance may be demonstrated in either of the following ways:
The addition shall comply with the applicable requirements, or
The addition, together with the entire existing building, shall comply with the requirements of this Code that shall apply to the entire building, as if it were a new building. The launch of the Energy Conservation and Sustainable Building Code (ECSBC) 2024 marks a significant advancement in India's commitment to sustainable development within the 

In [35]:
top[0].payload

{'heading': 'ECSBC 2024 > 2. Scope',
 'page': 42,
 'content_type': 'text',
 'source': 'ECBC_2024.pdf',
 'page_content': 'The Code is applicable to buildings or building complexes that have a connected load of 100 kW or greater or a contract demand of 120 kVA or greater and are intended to be used for commercial or office buildings.\nBuildings intended for residential purposes are not covered under this Code.',
 'metadata': {'heading': 'ECSBC 2024 > 2. Scope',
  'page': 42,
  'content_type': 'text',
  'source': 'ECBC_2024.pdf'}}

In [36]:
## test only dense
dense_retrival = _dense(query, None)

In [37]:
dense_retrival[0].payload['page_content']

'Table 8.9: Minimum Renewable Contribution towards meeting Contract Demand in ECSBC Plus Building'

In [38]:
scores = _bm25(query,None)

In [39]:
scores[0]

Hit(id='0ac73c41-2ced-485e-abea-2cba67ebd3a2', text='The Code is applicable to buildings or building complexes that have a connected load of 100 kW or greater or a contract demand of 120 kVA or greater and are intended to be used for commercial or office buildings.\nBuildings intended for residential purposes are not covered under this Code.', payload={'heading': 'ECSBC 2024 > 2. Scope', 'page': 42, 'content_type': 'text', 'source': 'ECBC_2024.pdf', 'page_content': 'The Code is applicable to buildings or building complexes that have a connected load of 100 kW or greater or a contract demand of 120 kVA or greater and are intended to be used for commercial or office buildings.\nBuildings intended for residential purposes are not covered under this Code.', 'metadata': {'heading': 'ECSBC 2024 > 2. Scope', 'page': 42, 'content_type': 'text', 'source': 'ECBC_2024.pdf'}}, dense_rank=None, bm25_rank=1, rrf_score=0.0, rerank_score=0.0)

## check the direct qdrant retrival

In [41]:
hybride_client = make_client("D:\\M.tech\\Research Area\\RAG pipeline\\DB_hybrid\\bge-small-en-v1.5_Embedding\\StructureAwareChunking")

Opened D:\M.tech\Research Area\RAG pipeline\DB_hybrid\bge-small-en-v1.5_Embedding\StructureAwareChunking; collections: ['bge_small_en_v1.5_StructureAwareChunking']


In [42]:
query = "Which standard does ECBC 2024 name as the reference for lighting levels, HVAC, thermal comfort and natural ventilation?"

In [43]:
dense_vec = embeder.encode(query, normalize_embeddings=True).tolist()

prefetch = [models.Prefetch(query=dense_vec, using="dense", limit=50)]


In [44]:
res = hybride_client.query_points(
    collection_name="bge_small_en_v1.5_StructureAwareChunking", prefetch=prefetch,
    query=models.RrfQuery(rrf=models.Rrf(k=60)),
    limit=30, with_payload=True).points

In [47]:
len(res)

30

In [62]:
text = " ".join(res[i].payload['page_content'] for i in range(len(res)))


In [63]:
text

"The launch of the Energy Conservation and Sustainable Building Code (ECSBC) 2024 marks a significant advancement in India's commitment to sustainable development within the building sector. Developed through a collaborative effort initiated in 2022, this code integrates energy efficiency with sustainability, reflecting the contributions of dedicated Committee Members.\nECSBC 2024 marks a significant evolution from its predecessor, ECBC 2017. While the energy efficiency thresholds have been slightly tightened, reflecting advancements in technology and research in building materials and equipment, the most notable addition is the inclusion of sustainability measures. These new elements aim to ensure that buildings not only conserve energy but also minimize their overall environmental impact throughout their life cycle. This approach balances economic growth with ecological responsibility, creating a foundation for long-term sustainable development.\nThe energy efficiency standards in EC

In [64]:
def rerank(query, hits, top_k):
    if not hits:
        return []
    scores = reranker.predict([(query, h.payload['page_content']) for h in hits], batch_size=16, show_progress_bar=False)
    for h, s in zip(hits, scores):
        h.rerank_score = float(s)
    return sorted(hits, key=lambda h: h.rerank_score, reverse=True)[:top_k]

In [65]:
top = rerank(query, fused, 10)

In [68]:
top


[Hit(id='2d18e354-0fd4-4015-acf5-1ea03df12450', text="The launch of the Energy Conservation and Sustainable Building Code (ECSBC) 2024 marks a significant advancement in India's commitment to sustainable development within the building sector. Developed through a collaborative effort initiated in 2022, this code integrates energy efficiency with sustainability, reflecting the contributions of dedicated Committee Members.\nECSBC 2024 marks a significant evolution from its predecessor, ECBC 2017. While the energy efficiency thresholds have been slightly tightened, reflecting advancements in technology and research in building materials and equipment, the most notable addition is the inclusion of sustainability measures. These new elements aim to ensure that buildings not only conserve energy but also minimize their overall environmental impact throughout their life cycle. This approach balances economic growth with ecological responsibility, creating a foundation for long-term sustainabl